# ETL Silver - GEFS c00 calibrado contra TIGGE cf (relleno 2000-01 -> 2006-09)

Corrida unica e idempotente (Decision 053), no un task del job diario. GEFS Reforecast v12 es un
archivo historico congelado (2000-01-01 -> 2019-12-31, sin dias nuevos); TIGGE `cf` recien empieza
en 2006-10-01. Este notebook mide el sesgo real de GEFS `c00` (unico miembro confiable -- los
perturbados `p01`-`p10` estan rotos en el Volume desde 2018-01-04, sin dispersion de ensemble
utilizable, ver Decision 053) contra `cf` en el solapamiento real 2006-10 -> 2019-12, por
sub-cuenca y lead_day, y aplica esa correccion **a cada punto de la grilla** (no a un promedio)
para producir `weather.silver.gefs_cf_fill_grid`: mismo shape que
`weather.bronze.ecmwf_forecast_cf`, solo para los dias que `cf` no tiene (< 2006-10-01).
`ETL_Silver_ECMWF_Subcuenca` la une a Bronze `cf` antes de agregar (modelo='cf'), asi que ni esa
agregacion ni Gold necesitan saber que una parte del historico viene de otra fuente -- y si mas
adelante cambia la metrica de agregacion (media -> P90 u otra), se recalcula igual para las dos
epocas sin tocar esta calibracion.

No hay sustituto para `pf`: sin miembros perturbados confiables en GEFS no hay con que calibrar
dispersion de ensemble.

In [ ]:
from pyspark.sql import functions as F, Window

GEFS_BRONZE = 'weather.bronze.gefs_reforecast'
CF_BRONZE = 'weather.bronze.ecmwf_forecast_cf'
MAPA_TABLE = 'weather.silver.punto_subcuenca'
BIAS_TABLE = 'weather.silver.gefs_cf_bias'
FILL_TABLE = 'weather.silver.gefs_cf_fill_grid'

GEFS_MEMBER = 'c00'
TIGGE_CF_START = '2006-10-01'
GEFS_LATEST_DATE = '2019-12-31'
LEAD_DAYS = list(range(1, 16))  # mismos 15 que FORECAST_LEAD_DAYS en ETL_Gold_Training_Dataset_v0

mapa = spark.table(MAPA_TABLE).select('latitude', 'longitude', 'subcuenca_id', 'subcuenca_nombre')
print(f'{MAPA_TABLE}: {mapa.count()} puntos')

In [ ]:
def grid_diario(bronze_table, filtro_extra=None):
    """Lee `bronze_table` (mismo shape que weather.bronze.ecmwf_forecast_cf: run_date, run_time,
    step_hours, latitude, longitude, tp_mm acumulado), lo tagea contra `punto_subcuenca` y
    convierte el acumulado a lluvia diaria por lead_day -- mismo patron de Window+lag que
    forecast_lead_day_features() en ETL_Gold_Training_Dataset_v0, pero a nivel de punto de
    grilla, no de sub-cuenca (Decision 053: el promedio a sub-cuenca es del calculo de
    Silver->Gold, no de esta calibracion -- ver el join a `mapa` mas abajo, que solo sirve para
    poder agrupar por sub-cuenca y calcular el sesgo, no para colapsar los puntos).

    Bronze trae pasos cada 3h/6h (GEFS) o cada 24h (cf); quedarse solo con los multiplos de 24h
    da los mismos 16 pasos (0..360h) en las dos fuentes -- confirmado que Bronze GEFS c00 tiene
    datos hasta step 384h, cubre de sobra hasta t+15.
    """
    src = spark.table(bronze_table).filter(F.col('step_hours') % 24 == 0)
    if filtro_extra is not None:
        src = src.filter(filtro_extra)
    src = src.select('run_date', 'run_time', 'step_hours', 'latitude', 'longitude', 'tp_mm')

    tagged = src.join(mapa, on=['latitude', 'longitude'], how='inner')

    w = Window.partitionBy('run_date', 'run_time', 'latitude', 'longitude').orderBy('step_hours')
    diario = (
        tagged
        .withColumn('tp_acum_prev', F.lag('tp_mm').over(w))
        .filter(F.col('step_hours') > 0)
        .withColumn('tp_dia_mm', F.col('tp_mm') - F.coalesce(F.col('tp_acum_prev'), F.lit(0.0)))
        .withColumn('lead_day', (F.col('step_hours') / F.lit(24)).cast('int'))
        .filter(F.col('lead_day').isin(LEAD_DAYS))
    )
    return diario


# Serverless no soporta CACHE/PERSIST TABLE (NOT_SUPPORTED_WITH_SERVERLESS), asi que
# gefs_diario se recalcula dos veces (bias y aplicacion) en vez de cachearse.
gefs_diario = grid_diario(GEFS_BRONZE, F.col('member') == F.lit(GEFS_MEMBER))
# GEFS no tiene dias nuevos; recortar cf a su rango evita procesar 2020+ sin ninguna ganancia
# (el join con el solapamiento lo descartaria igual, esto solo ahorra el escaneo).
cf_diario = grid_diario(CF_BRONZE, F.col('run_date') <= F.lit(GEFS_LATEST_DATE))

In [ ]:
gefs_sub = gefs_diario.groupBy('run_date', 'subcuenca_nombre', 'lead_day').agg(F.avg('tp_dia_mm').alias('gefs_mm'))
cf_sub = cf_diario.groupBy('run_date', 'subcuenca_nombre', 'lead_day').agg(F.avg('tp_dia_mm').alias('cf_mm'))

# El solapamiento real: mismo run_date con dato en las dos fuentes. Antes de la Decision 034
# esto daba 0 dias (Decision 035); con GEFS c00 completo 2000-2019 ya hay volumen de sobra.
solapamiento = gefs_sub.join(cf_sub, on=['run_date', 'subcuenca_nombre', 'lead_day'], how='inner')

bias_observado = (
    solapamiento
    .groupBy('subcuenca_nombre', 'lead_day')
    .agg(
        F.avg(F.col('cf_mm') - F.col('gefs_mm')).alias('bias_mm'),
        F.countDistinct('run_date').alias('n_dias_solapamiento'),
    )
    .withColumn('calibrado', F.lit(True))
)

combos = (
    mapa.select('subcuenca_nombre').distinct()
    .crossJoin(spark.createDataFrame([(d,) for d in LEAD_DAYS], ['lead_day']))
)

bias_table = (
    combos
    .join(bias_observado, on=['subcuenca_nombre', 'lead_day'], how='left')
    # Sin evidencia real para un combo (subcuenca, lead_day) no se inventa una correccion --
    # mismo criterio que apply_bias() del prototipo offline (notebooks_local/forecast_calibration).
    .withColumn('bias_mm', F.coalesce(F.col('bias_mm'), F.lit(0.0)))
    .withColumn('n_dias_solapamiento', F.coalesce(F.col('n_dias_solapamiento'), F.lit(0).cast('bigint')))
    .withColumn('metodo', F.lit('aditivo'))
    .withColumn('calibrado', F.coalesce(F.col('calibrado'), F.lit(False)))
    .withColumn('computed_at', F.current_timestamp())
    .select('subcuenca_nombre', 'lead_day', 'bias_mm', 'n_dias_solapamiento', 'metodo', 'calibrado', 'computed_at')
)

bias_table.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(BIAS_TABLE)
n_bias = spark.table(BIAS_TABLE).count()
print(f'{BIAS_TABLE}: {n_bias} filas')
spark.table(BIAS_TABLE).orderBy('subcuenca_nombre', 'lead_day').show(60, truncate=False)

In [ ]:
bias_lookup = spark.table(BIAS_TABLE).select('subcuenca_nombre', 'lead_day', 'bias_mm')

calibrado_diario = (
    gefs_diario
    .join(bias_lookup, on=['subcuenca_nombre', 'lead_day'], how='left')
    .withColumn('bias_mm', F.coalesce(F.col('bias_mm'), F.lit(0.0)))
    .withColumn('tp_dia_mm_calibrado', F.greatest(F.col('tp_dia_mm') + F.col('bias_mm'), F.lit(0.0)))
)

# Se vuelve a acumular por punto (cumsum de la lluvia diaria calibrada) para reconstruir la misma
# convencion de tp_mm acumulado-desde-el-inicio-de-la-corrida que usa cf real -- asi el consumidor
# (ETL_Silver_ECMWF_Subcuenca) no necesita saber que este dia viene de GEFS.
w_cum = Window.partitionBy('run_date', 'run_time', 'latitude', 'longitude').orderBy('lead_day')

fill_grid = (
    calibrado_diario
    .withColumn('tp_mm', F.sum('tp_dia_mm_calibrado').over(w_cum))
    .withColumn('step_hours', (F.col('lead_day') * F.lit(24)).cast('int'))
    .withColumn('valid_date', F.date_add(F.col('run_date'), F.col('lead_day')))
    .withColumn('valid_datetime', F.to_timestamp(F.col('valid_date')))
    .withColumn('number', F.lit(None).cast('int'))
    .withColumn('tipo', F.lit('cf'))
    .withColumn('source_api', F.lit('gefs_reforecast_v12_calibrado'))
    .withColumn('source_file', F.lit(None).cast('string'))
    .withColumn('extracted_at', F.current_timestamp())
    .withColumn('ingestion_date', F.current_date())
    .withColumn('loaded_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .withColumn('fuente', F.lit('gefs_calibrado'))
    .withColumnRenamed('bias_mm', 'bias_mm_aplicado')
    # Alcance explicito (Decision 053): solo llena lo que cf no tiene. Nunca compite con un dia
    # real -- Bronze cf empieza justo en TIGGE_CF_START, asi que esto no puede solaparse.
    .filter(F.col('run_date') < F.lit(TIGGE_CF_START))
    .select(
        'run_date', 'run_time', 'step_hours', 'valid_date', 'valid_datetime',
        'latitude', 'longitude', 'number', 'tp_mm', 'tipo', 'source_api', 'source_file',
        'extracted_at', 'ingestion_date', 'loaded_at', 'updated_at', 'fuente', 'bias_mm_aplicado',
    )
)

fill_grid.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(FILL_TABLE)

resumen = spark.table(FILL_TABLE).agg(
    F.min('run_date').alias('desde'),
    F.max('run_date').alias('hasta'),
    F.countDistinct('run_date').alias('dias'),
    F.countDistinct(F.concat_ws('_', 'latitude', 'longitude')).alias('puntos'),
    F.count(F.lit(1)).alias('filas'),
).collect()[0]
print(f'{FILL_TABLE}: {resumen["filas"]} filas, {resumen["dias"]} dias '
      f'({resumen["desde"]} .. {resumen["hasta"]}), {resumen["puntos"]} puntos de grilla distintos')